# Réalisation de modèle de régression sur les données Sanitoral du P7

## Récupérartion des données

Librairie **sklearn** à installer

In [1]:
import pandas as pd

In [2]:
! pip install scikit-learn

In [3]:
# Lecrure du CSV - attention traitement des dates !

df = pd.read_csv('data.csv', parse_dates=['Start Date'])
df['Start Date'] = df['Start Date'].astype('int64') // 10**9

df

,Unnamed: 0,Project ID,Phase,Start Date,Planned_Duration,Planned_Cost,Planned_Delivrable,Project,Actual_Duration,_merge,...,Phase 1 - Planning,Phase 2 - Initiation,Phase 3 - Implementation,Phase 4 - Manufacturing,Phase A - Initiation,Phase B - Preparation,Phase C - Development,Phase D - Testing,Phase E - Deployment,Phase F - Post-deployment
0,0,1,Phase 1 - Planning,1516060800,196,50000,10,1,196,both,...,1,0,0,0,0,0,0,0,0,0
1,1,1,Phase 2 - Initiation,1517356800,15,100000,17,1,7,both,...,0,1,0,0,0,0,0,0,0,0
2,2,1,Phase 3 - Implementation,1523923200,197,150000,26,1,176,both,...,0,0,1,0,0,0,0,0,0,0
3,3,1,Phase 4 - Manufacturing,1539129600,61,450000,23,1,64,both,...,0,0,0,1,0,0,0,0,0,0
4,4,2,Phase 1 - Planning,1517270400,16,100000,6,2,16,both,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515,515,103,Phase F - Post-deployment,1618963200,167,1400,24,103,171,both,...,0,0,0,0,0,0,0,0,0,1
516,516,104,Phase 1 - Planning,1526083200,186,50000,21,104,163,both,...,1,0,0,0,0,0,0,0,0,0
517,517,104,Phase 2 - Initiation,1528329600,185,100000,27,104,179,both,...,0,1,0,0,0,0,0,0,0,0
518,518,104,Phase 3 - Implementation,1529712000,72,150000,17,104,168,both,...,0,0,1,0,0,0,0,0,0,0


In [4]:
df.dtypes

Unnamed: 0                     int64
Project ID                     int64
Phase                         object
Start Date                     int64
Planned_Duration               int64
Planned_Cost                   int64
Planned_Delivrable             int64
Project                        int64
Actual_Duration                int64
_merge                        object
Delay                          int64
Is_Late                        int64
Delay_Category                object
Start_Month                    int64
Start_Quarter                  int64
Start_Year                     int64
Planned_Intensity            float64
Project Type                  object
Country                       object
Region                        object
Type                          object
Phase 1 - Planning             int64
Phase 2 - Initiation           int64
Phase 3 - Implementation       int64
Phase 4 - Manufacturing        int64
Phase A - Initiation           int64
Phase B - Preparation          int64
P

## Identification de "X" et "y", séparation en données train / test

Création de df_X et df_y, selection des colonnes  
Utilisation de méthode '**train_test_split**' pour créer les jeu train et test  
Doc : https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

In [5]:
#df['Start Date'] = df_X['Start Date'].astype('int64') // 10**9

df_X = df[['Start Date', 'Planned_Duration',
       'Planned_Delivrable', 'Actual_Duration',
       'Planned_Intensity',
       'Phase 1 - Planning', 'Phase 2 - Initiation',
       'Phase 3 - Implementation', 'Phase 4 - Manufacturing',
       'Phase A - Initiation', 'Phase B - Preparation',
       'Phase C - Development', 'Phase D - Testing', 'Phase E - Deployment',
       'Phase F - Post-deployment']]
df_X.head()

,Start Date,Planned_Duration,Planned_Delivrable,Actual_Duration,Planned_Intensity,Phase 1 - Planning,Phase 2 - Initiation,Phase 3 - Implementation,Phase 4 - Manufacturing,Phase A - Initiation,Phase B - Preparation,Phase C - Development,Phase D - Testing,Phase E - Deployment,Phase F - Post-deployment
0,1516060800,196,10,196,1.530612,1,0,0,0,0,0,0,0,0,0
1,1517356800,15,17,7,34.000000,0,1,0,0,0,0,0,0,0,0
2,1523923200,197,26,176,3.959391,0,0,1,0,0,0,0,0,0,0
3,1539129600,61,23,64,11.311475,0,0,0,1,0,0,0,0,0,0
4,1517270400,16,6,16,11.250000,1,0,0,0,0,0,0,0,0,0


In [6]:
df_y = df['Planned_Cost'].ravel()
df_y

array([ 50000, 100000, 150000, 450000, 100000, 200000, 300000, 900000,
        50000, 100000, 150000, 450000,  50000, 100000, 150000, 450000,
        50000, 100000, 150000, 450000,  50000, 100000, 150000, 450000,
       100000, 200000, 300000, 900000, 100000, 200000, 300000, 900000,
       100000, 200000, 300000, 900000,  33300,  66700, 100000, 300000,
       100000, 200000, 300000, 900000,  50000, 100000, 150000, 450000,
        50000, 100000, 150000, 450000,  50000, 100000, 150000, 450000,
        33300,  66700, 100000, 300000,  33300,  66700, 100000, 300000,
       100000, 200000, 300000, 900000,  33300,  66700, 100000, 300000,
       100000, 200000, 300000, 900000,  50000, 100000, 150000, 450000,
        50000, 100000, 150000, 450000,  50000, 100000, 150000, 450000,
       100000, 200000, 300000, 900000,  50000, 100000, 150000, 450000,
        33300,  66700, 100000, 300000, 100000, 200000, 300000, 900000,
        50000, 100000, 150000, 450000, 100000, 200000, 300000, 900000,
      

In [7]:
from sklearn.model_selection import train_test_split

# Séparation en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(
    df_X, df_y, test_size=0.2, random_state=42
)

# doc : https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

print('X_train', X_train.shape)
print('X_test', X_test.shape)
print('y_train', y_train.shape)
print('y_test', y_test.shape)


X_train (416, 15)
X_test (104, 15)
y_train (416,)
y_test (104,)


## Création de modèle de régression très simple - baseline

**DummyRegressor** est un modèle tout bête  
Doc : https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html


Régresseur simple qui fait des prédictions sans utiliser les features d'entrée. Il sert de modèle de base (baseline) pour comparer la performance d'autres modèles plus complexes.  
**Principe** :  
Ignorer les features : Il ne regarde pas les variables d'entrée (X) pour faire ses prédictions.  
Stratégie simple : Il utilise une règle de base sur les valeurs cibles (y) d'entraînement.  
Référence : Il établit un score "minimum" à battre par de vrais modèles.  
Stratégies disponibles (strategy) :  
mean (défaut) : Prédit toujours la moyenne des valeurs d'entraînement  
median : Prédit toujours la médiane  
quantile : Prédit un quantile spécifique  
constant : Prédit une valeur constante que vous fournissez  

Pourquoi l'utiliser ?  
Établir une baseline : Si vos modèles complexes ne font pas mieux que le DummyRegressor, ils ne sont pas performants.  
Vérifier l'utilité des features : Cela confirme que les variables d'entrée apportent réellement de l'information.  
Détecter des problèmes : Un modèle qui performe moins bien qu'une prédiction constante indique un problème.  

In [8]:
from sklearn.dummy import DummyRegressor

dummy_regr = DummyRegressor(strategy="mean")

DummyRegressor()

DummyRegressor()

## Entrainement modèle

Utilisation de la méthode **fit**()

Méthode fondamentale qui entraîne un modèle sur des données. Elle est présente sur tous les estimateurs (modèles) de scikit-learn.  
**Principe** :  
Apprentissage : Le modèle apprend les patterns, relations et structures à partir des données fournies.  
Ajustement des paramètres internes : Elle calcule et stocke les coefficients, poids ou autres paramètres internes du modèle.  
État persistant : Après fit(), le modèle est "entraîné" et prêt à faire des prédictions avec predict().  

In [9]:
dummy_regr.fit(X_train, y_train)

DummyRegressor()

## Affichage score R2

Score R2 : https://scikit-learn.org/stable/modules/model_evaluation.html#r2-score-the-coefficient-of-determination  
Score MAE : https://scikit-learn.org/stable/modules/model_evaluation.html#mean-absolute-error  
Score MSE : https://scikit-learn.org/stable/modules/model_evaluation.html#mean-squared-error

**R² (Coefficient de détermination)**  
Proportion de variance de la variable cible qui est expliquée par le modèle.  
Interprétation :  
1.0 : Le modèle prédit parfaitement  
0.0 : Le modèle ne fait pas mieux qu'une prédiction constante (la moyenne)  
Négatif : Le modèle est pire que la moyenne (mauvais signe !)  
Formule : R² = 1 - (Somme des carrés des résidus / Variance totale)  
Avantage : Interprétation intuitive (pourcentage de variance expliquée)  

**MSE (Mean Squared Error)**  
Moyenne des carrés des erreurs (différences entre prédictions et valeurs réelles).  
Formule : MSE = (1/n) * Σ(y_true - y_pred)²  
Caractéristiques :  
Pénalise lourdement les grandes erreurs (à cause du carré)  
Toujours positive (0 = prédiction parfaite)  
Unité : au carré de l'unité de la cible (difficile à interpréter directement)  
Avantage : Sensible aux outliers (les détecte facilement)  

**MAE (Mean Absolute Error)**  
Moyenne des valeurs absolues des erreurs.  
Formule : MAE = (1/n) * Σ|y_true - y_pred|  
Caractéristiques :  
Traite toutes les erreurs de manière égale (pas de pénalisation excessive)  
Toujours positive (0 = prédiction parfaite)  
Unité : identique à l'unité de la cible (facile à interpréter)  
Avantage : Interprétation intuitive ("l'erreur moyenne est de X unités")  



In [10]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

y_pred_train = dummy_regr.predict(X_train)
y_pred_test = dummy_regr.predict(X_test)

# Évaluation du modèle
print("=== SCORES DU MODÈLE ===")
print(f"Score R² (entraînement) : {dummy_regr.score(X_train, y_train):.3f}")
print(f"Score R² (test) : {dummy_regr.score(X_test, y_test):.3f}")

print(f"MSE (test) : {mean_squared_error(y_test, y_pred_test):.3f}")
print(f"RMSE (test) : {np.sqrt(mean_squared_error(y_test, y_pred_test)):.3f}")
print(f"MAE (test) : {mean_absolute_error(y_test, y_pred_test):.3f}")

=== SCORES DU MODÈLE ===
Score R² (entraînement) : 0.000
Score R² (test) : -0.002
MSE (test) : 26544488943.752
RMSE (test) : 162924.795
MAE (test) : 111305.649


# Exercice : Recommencer avec un modèle plus efficace

Utiliser un modèle plus efficace : **LinearRegression**  
doc : https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

Algorithme d'apprentissage supervisé utilisé pour la régression (prédire une valeur numérique continue) basé sur une relation linéaire.  
**Principe** :  
Relation linéaire : Il cherche à modéliser la relation entre les variables d'entrée (features) et la variable cible par une équation linéaire : y = Ax + B.  
Moindres carrés ordinaires : Il trouve la ligne (ou hyperplan) qui minimise la somme des carrés des erreurs (différences entre prédictions et valeurs réelles).  
Coefficients : Il calcule un coefficient (poids) pour chaque feature, indiquant son importance et sa relation (positive ou négative) avec la cible.  

Simplicité : Facile à comprendre et à interpréter.  
Rapidité : Entraînement et prédictions très rapides.  
Interprétabilité : Les coefficients montrent clairement l'impact de chaque variable.  

**Limites** :  
Suppose une relation linéaire entre les features et la cible.  
Sensible aux outliers (valeurs aberrantes).  
Peut être impacté par la multicolinéarité (features fortement corrélées entre elles).  

In [11]:
from sklearn.linear_model import LinearRegression

modele = LinearRegression()
modele.fit(X_train, y_train)

y_pred_train = modele.predict(X_train)
y_pred_test = modele.predict(X_test)

# Évaluation du modèle
print("=== SCORES DU MODÈLE LinearRegression ===")
print(f"Score R² (entraînement) : {modele.score(X_train, y_train):.3f}")
print(f"Score R² (test) : {modele.score(X_test, y_test):.3f}")


=== SCORES DU MODÈLE LinearRegression ===
Score R² (entraînement) : 0.822
Score R² (test) : 0.765


## Utilisation de modèle RandomForestRegressor

Doc : https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html#randomforestregressor

C'est un algorithme d'apprentissage supervisé utilisé pour la régression (prédire une valeur numérique continue).  
**principe** :  
Forêt aléatoire : Il construit de nombreux arbres de décision.  
Bagging : Chaque arbre est entraîné sur un échantillon aléatoire différent des données d'origine (avec remise).  
Randomisation : Pour chaque division dans un arbre, il ne considère qu'un sous-ensemble aléatoire des caractéristiques.  
Prédiction : La prédiction finale est la moyenne des prédictions de tous les arbres individuels.  

Robuste : Très performant, il réduit le surapprentissage (overfitting) par rapport à un seul arbre de décision.  
Puissant : Gère bien les relations non-linéaires et les interactions complexes.  
Pratique : Nécessite peu de pré-traitement des données (pas besoin de normalisation).  

In [12]:
from sklearn.ensemble import RandomForestRegressor

modele = RandomForestRegressor()
modele.fit(X_train, y_train)

y_pred_train = modele.predict(X_train)
y_pred_test = modele.predict(X_test)

# Évaluation du modèle
print("=== SCORES DU MODÈLE RandomForestRegressor ===")
print(f"Score R² (entraînement) : {modele.score(X_train, y_train):.3f}")
print(f"Score R² (test) : {modele.score(X_test, y_test):.3f}")


=== SCORES DU MODÈLE RandomForestRegressor ===
Score R² (entraînement) : 0.975
Score R² (test) : 0.723


## Optimisation du modèle - GridSearchCV

Doc : https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

Outil de recherche d'hyperparamètres qui permet de trouver automatiquement les meilleurs réglages pour un modèle.  
**Principe** :  
Grille de paramètres : Vous définissez une grille (un dictionnaire) avec les hyperparamètres à tester et les valeurs possibles.  
Recherche exhaustive : Il teste toutes les combinaisons possibles de ces paramètres.  
Validation croisée (CV) : Pour chaque combinaison, il évalue la performance du modèle en utilisant la validation croisée (k-fold) pour éviter le surapprentissage.  
Sélection : Il conserve la combinaison qui donne les meilleures performances.  
 
Automatisation : Évite de tester manuellement des dizaines de combinaisons.  
Optimisation : Garantit de trouver la meilleure combinaison parmi celles que vous avez spécifiées.  
Robustesse : La validation croisée donne une évaluation fiable des performances.  

In [13]:
from sklearn.model_selection import GridSearchCV

# Définition de la grille de paramètres à tester
param_grid = {
    'n_estimators': [50, 100, 200],  # Nombre d'arbres
    'max_depth': [5, 10, 15, None],  # Profondeur maximale des arbres
    'min_samples_split': [2, 5, 10],  # Échantillons minimum pour split
    'min_samples_leaf': [1, 2, 4],     # Échantillons minimum par feuille
    'max_features': ['sqrt', 'log2']   # Nombre de features considérées
}

# Création du modèle avec GridSearch
rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(
    rf, 
    param_grid, 
    cv=5,  # Validation croisée à 5 folds
    scoring='r2',
    n_jobs=-1,  # Utilise tous les coeurs CPU
    verbose=1
)

# Recherche des meilleurs paramètres (peut prendre du temps)
print("Recherche des meilleurs paramètres en cours...")
grid_search.fit(X_train, y_train)

print(f"\nMeilleurs paramètres trouvés :")
print(grid_search.best_params_)
print(f"Meilleur score cross-validation : {grid_search.best_score_:.3f}\n")

# Évaluation sur le test
best_rf = grid_search.best_estimator_
print(f"Score R² sur train avec modèle optimisé : {best_rf.score(X_train, y_train):.3f}")
print(f"Score R² sur test avec modèle optimisé : {best_rf.score(X_test, y_test):.3f}")

Recherche des meilleurs paramètres en cours...
Fitting 5 folds for each of 216 candidates, totalling 1080 fits

Meilleurs paramètres trouvés :
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 100}
Meilleur score cross-validation : 0.768

Score R² sur train avec modèle optimisé : 0.873
Score R² sur test avec modèle optimisé : 0.794


## Sauvegardes

Le modèle lui même doit être sauvegardé (pour utilisation dans une application).  
Les données sont aussi sauvegardées en CSV pour utilisation.

In [14]:
import joblib
# modèle lui-même
joblib.dump(best_rf, 'rf_modele.pkl')

y_pred = best_rf.predict(df_X)

resultats = pd.DataFrame({
    'Project ID': df['Project ID'],  
    'Phase': df['Phase'],      # À remplacer par la vraie colonne si disponible
    'Planned_Cost': df_y,
    'Planned_Cost_predicted': y_pred.flatten()  # Prédictions
})

resultats.to_csv('results.csv')
resultats

,Project ID,Phase,Planned_Cost,Planned_Cost_predicted
0,1,Phase 1 - Planning,50000,86262.622643
1,1,Phase 2 - Initiation,100000,137903.278701
2,1,Phase 3 - Implementation,150000,179302.196131
3,1,Phase 4 - Manufacturing,450000,546330.912143
4,2,Phase 1 - Planning,100000,78034.558227
...,...,...,...,...
515,103,Phase F - Post-deployment,1400,9300.908500
516,104,Phase 1 - Planning,50000,86680.640610
517,104,Phase 2 - Initiation,100000,128101.142930
518,104,Phase 3 - Implementation,150000,218556.494747
